# Day 7: Deploy your first LLM-powered app on Streamlit Cloud and integrate basic logging

## 🧠 Core Theory (Just-in-Time)

As you transition into AI Engineering, getting your models out of local scripts and into a usable interface is critical. 

### Why Streamlit Cloud?
Streamlit Cloud allows rapid deployment of Python-based UIs directly from a GitHub repository. It abstracts away server management, Dockerization, and reverse proxies, allowing you to focus purely on the application logic. For AI prototypes and internal tooling, it is the industry standard for fast iteration.

### Why Structured Logging?
In software engineering, `print()` statements are insufficient for production. In AI Engineering, this is doubly true. LLMs are non-deterministic, meaning the same input can yield different outputs. 
- **Traceability:** Logging captures the exact prompts sent and completions received.
- **Observability:** If an API call to OpenAI fails or times out, structured logging (e.g., using Python's built-in `logging` module) ensures you have timestamps, severity levels (INFO, WARNING, ERROR), and module names to diagnose the failure quickly without exposing raw stack traces to the end-user.

### AI Security Implications
When deploying LLM applications, you must proactively defend against common vulnerabilities:
- **PII Protection:** Never log raw user inputs indefinitely if they might contain Personally Identifiable Information (PII). Always truncate, anonymize, or omit sensitive data in logs.
- **Prompt Injection:** Treat all user input as untrusted. While basic chatbots are less susceptible than agents with tools, you should still log anomalies and use system prompts to constrain behavior.
- **Fallback Mechanisms:** API outages happen. If your LLM provider is down, your app should gracefully degrade, providing a polite fallback response rather than a generic error or raw stack trace.


## 💻 Code Implementation

Below is a tiered progression of Python code examples showing how to integrate Streamlit and LangChain, from a basic script to a production-grade application.

### 1. Basic: Isolating the Core Concept
This example shows the absolute minimum boilerplate required to get an LLM response into a Streamlit UI.


In [1]:
import os
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def basic_app():
    """Minimal Streamlit LLM implementation."""
    st.title("Basic LLM App")
    
    api_key = os.getenv("OPENAI_API_KEY", "")
    if not api_key:
        st.warning("Please set OPENAI_API_KEY environment variable.")
        return
        
    llm = ChatOpenAI(model="gpt-3.5-turbo", api_key=api_key)
    
    user_input = st.text_input("Ask a question:")
    if user_input:
        try:
            response = llm.invoke([HumanMessage(content=user_input)])
            st.write(response.content)
        except Exception as e:
            st.error(f"An error occurred: {e}")
            # Fallback for notebook validation
            st.write("Fallback response due to missing real API key.")

# if __name__ == '__main__':
#     basic_app()


### 2. Medium: Clean OOP and State Management
Here we introduce Object-Oriented Programming (OOP) to cleanly manage `st.session_state` and object interactions. This pattern keeps the global namespace clean and makes testing easier.

In [2]:
import os
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

class ChatStateManager:
    """Manages Streamlit session state for the chat history."""
    
    def __init__(self, session_key: str = "chat_history"):
        self.session_key = session_key
        if self.session_key not in st.session_state:
            st.session_state[self.session_key] = []
            
    def get_history(self) -> list:
        return st.session_state[self.session_key]
        
    def add_message(self, message) -> None:
        st.session_state[self.session_key].append(message)


class MediumChatApp:
    """A chat application demonstrating OOP principles and state management."""
    
    def __init__(self):
        self.state_manager = ChatStateManager()
        self.api_key = os.getenv("OPENAI_API_KEY", "")
        if self.api_key:
            self.llm = ChatOpenAI(model="gpt-3.5-turbo", api_key=self.api_key)
        else:
            self.llm = None
            
    def render_history(self):
        """Renders existing chat history to the UI."""
        for msg in self.state_manager.get_history():
            role = "user" if isinstance(msg, HumanMessage) else "assistant"
            st.chat_message(role).write(msg.content)

    def run(self):
        st.title("Medium LLM Chat App (OOP)")
        
        if not self.llm:
            st.warning("Please set OPENAI_API_KEY environment variable.")
            return
            
        self.render_history()
        
        if prompt := st.chat_input("Say something"):
            print(f"[LOG] User asked: {prompt}")
            
            st.chat_message("user").write(prompt)
            user_msg = HumanMessage(content=prompt)
            self.state_manager.add_message(user_msg)
            
            try:
                response = self.llm.invoke(self.state_manager.get_history())
                st.chat_message("assistant").write(response.content)
                self.state_manager.add_message(response)
                print("[LOG] LLM replied successfully.")
            except Exception as e:
                error_msg = f"API Error: {e}"
                print(f"[ERROR] {error_msg}")
                st.error("Something went wrong connecting to the AI.")
                
                # Fallback mechanism
                fallback = AIMessage(content="Fallback response due to missing real API key or outage.")
                st.chat_message("assistant").write(fallback.content)
                self.state_manager.add_message(fallback)

# if __name__ == '__main__':
#     app = MediumChatApp()
#     app.run()


### 3. Advanced: Production-Grade Implementation
Below is a production-grade Streamlit application. It emphasizes strict type hinting, proper exception handling, and robust structured logging using Python's `logging` module.


In [3]:
import logging
import os
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage

# 1. Configure Logging (Production-Grade Setup)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("app.streamlit.llm")

class ChatBotService:
    """Service class handling LLM interactions with strict type hinting and fallbacks."""
    
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.model_name = model_name
        self.llm = self._initialize_llm()
        
    def _initialize_llm(self) -> ChatOpenAI | None:
        """Initializes the ChatOpenAI client with proper API key resolution."""
        api_key: str | None = os.getenv("OPENAI_API_KEY")
        if not api_key:
            try:
                api_key = st.secrets.get("OPENAI_API_KEY")
            except FileNotFoundError:
                api_key = None

        if not api_key:
            logger.error("OPENAI_API_KEY is missing from environment and secrets.")
            return None
        
        try:
            llm = ChatOpenAI(
                model=self.model_name,
                temperature=0.7,
                api_key=api_key,
                request_timeout=10.0
            )
            logger.info("ChatOpenAI client initialized successfully.")
            return llm
        except Exception as e:
            logger.error(f"Failed to initialize ChatOpenAI: {e}")
            return None

    def generate_response(self, user_prompt: str) -> str:
        """Generates a response with secure logging and fallback mechanisms."""
        # PII Protection: Truncate prompt in logs to avoid storing sensitive user data
        logger.info(f"Received user prompt: {user_prompt[:50]}...") 
        
        if not self.llm:
            return "Application configuration error. Check logs for details."
            
        messages: list[BaseMessage] = [
            SystemMessage(content="You are a helpful AI assistant. Always prioritize safety and accuracy."),
            HumanMessage(content=user_prompt)
        ]
        
        try:
            response = self.llm.invoke(messages)
            logger.info("Successfully generated response from LLM.")
            return str(response.content)
        except Exception as e:
            logger.error(f"Error during LLM invocation: {e}")
            # Secure Fallback: Do not expose raw stack traces to the user
            return "I am currently experiencing technical difficulties. Please try again later."


class AdvancedChatApp:
    """Production-grade Streamlit application integrating the ChatBotService."""
    
    def __init__(self):
        self.bot_service = ChatBotService()
        
    def initialize_session(self) -> None:
        if "messages" not in st.session_state:
            st.session_state.messages = []
            
    def display_history(self) -> None:
        for message in st.session_state.messages:
            with st.chat_message(message["role"]):
                st.markdown(message["content"])
                
    def handle_input(self) -> None:
        if user_prompt := st.chat_input("What is on your mind?"):
            st.chat_message("user").markdown(user_prompt)
            st.session_state.messages.append({"role": "user", "content": user_prompt})

            with st.chat_message("assistant"):
                with st.spinner("Thinking..."):
                    response_content = self.bot_service.generate_response(user_prompt)
                    st.markdown(response_content)
            
            st.session_state.messages.append({"role": "assistant", "content": response_content})
            
    def run(self) -> None:
        """Main Streamlit application entry point."""
        st.set_page_config(page_title="Day 7: Prod LLM App", page_icon="🤖")
        st.title("Enterprise LangChain & Streamlit Cloud App")
        
        if not self.bot_service.llm:
            st.error("System is unavailable. Please check backend configuration.")
            st.stop()
            
        self.initialize_session()
        self.display_history()
        self.handle_input()

if __name__ == "__main__":
    # Note: When running in a Jupyter Notebook, Streamlit apps won't render inline easily.
    # Extract this code to a separate file (e.g., app.py) to run it.
    pass
    # app = AdvancedChatApp()
    # app.run()


## 🛠️ Practical Lab / Homework

**Your Task:** Deploy the application above to Streamlit Cloud.

1. **Extract Code:** Create a new local directory. Inside it, create a file named `app.py` and paste the Advanced Python code from the cell above into it.
2. **Define Dependencies:** Create a `requirements.txt` file in the same directory. Add the following dependencies:
   ```text
   streamlit
   langchain-core
   langchain-openai
   ```
3. **Version Control:** Initialize a Git repository, commit `app.py` and `requirements.txt`, and push to a new public or private repository on GitHub.
4. **Deploy:** Go to [share.streamlit.io](https://share.streamlit.io), sign in with GitHub, and click "New app". Select your repository, branch, and `app.py` as the main file path.
5. **Configure Secrets:** Before the app finishes booting, go to the app's Settings -> Secrets on the Streamlit dashboard and add your API key:
   ```toml
   OPENAI_API_KEY="sk-..."
   ```
6. **Verify Logs:** Once deployed, interact with the app. Then, click "Manage app" in the bottom right corner of your Streamlit Cloud deployment to view the terminal logs. Verify that your `INFO` and `ERROR` logs are appearing correctly.
7. **Video Walkthrough:** Record a brief (2-3 minute) async video walkthrough (e.g., using Loom) of your application running in production. Explicitly discuss the design decisions you made, focusing on how you implemented clean OOP and state management.

---

## ⚠️ Common Pitfalls

When moving from local scripts to hosted Streamlit environments, watch out for:

1. **Hardcoding Secrets:** Storing API keys directly in `app.py` is a massive security risk, especially if pushed to GitHub. Always use `os.getenv()` or `st.secrets` as demonstrated.
2. **Ignoring UI Feedback:** AI API calls take time (latency). Failing to use `st.spinner()` or `st.write_stream()` makes the app feel frozen, leading users to spam the submit button.
3. **Uncontrolled Reruns:** Streamlit executes the entire script top-to-bottom on *every* user interaction. Failing to store chat history in `st.session_state` means the app will "forget" the conversation every time the user sends a new message.
4. **Silent Failures:** Without proper `try/except` blocks and logging, API errors (like rate limits or invalid keys) will either fail silently (freezing the app) or dump raw stack traces into the UI, exposing backend details to the user.


## 📚 Reference Links

- [Streamlit Official Documentation: Build a basic LLM chat app](https://docs.streamlit.io/knowledge-base/tutorials/build-conversational-apps)
- [Streamlit Cloud Deployment Guide](https://docs.streamlit.io/streamlit-community-cloud/deploy-your-app)
- [Python `logging` HOWTO](https://docs.python.org/3/howto/logging.html)
